# Inspección de trazados piloto

## Objetivo

Este notebook conserva la inspección **histórica del piloto**: verifica la estructura, el sistema de coordenadas, la geometría y la continuidad de sus trazados originales. No pretende mostrar las 22 rutas actuales. Para el inventario completo y sus estados vigentes, consultar `04_ampliacion_rutas.ipynb`.

Desde el 20 de octubre, las celdas de comprobación son de solo lectura: reconstruyen geometrías en memoria y contrastan los GeoJSON archivados sin reescribirlos. Las salidas de código se actualizaron ese día; el contenido sigue describiendo el piloto histórico.

Los archivos de `02_datos_originales` y `03_datos_intermedios` se leen sin modificarlos. Este notebook no genera productos nuevos.

## R001 – Sendero Cumbre Rucu Pichincha

El archivo original contiene los segmentos descargados desde OpenStreetMap mediante Overpass Turbo. En esta sección se comprobará que pueda leerse correctamente y que conserve las geometrías esperadas.

In [1]:
from pathlib import Path
import os
import sys

# Configurar los datos internos de GDAL antes de importar GeoPandas
gdal_data = Path(sys.prefix) / "Library" / "share" / "gdal"

if gdal_data.exists():
    os.environ["GDAL_DATA"] = str(gdal_data)

import geopandas as gpd

# Localizar la carpeta principal del proyecto
actual = Path.cwd().resolve()

if (actual / "01_documentacion").exists():
    proyecto = actual
elif (actual.parent / "01_documentacion").exists():
    proyecto = actual.parent
else:
    raise FileNotFoundError("No se encontró la carpeta principal del proyecto")

archivo_r001 = (
    proyecto
    / "02_datos_originales"
    / "trazados"
    / "R001_rucu_pichincha_osm_2026-09-30.geojson"
)

# Leer el archivo original sin modificarlo
trazado_r001 = gpd.read_file(archivo_r001)

print("Archivo:", archivo_r001.name)
print("Número de geometrías:", len(trazado_r001))
print("Sistema de coordenadas:", trazado_r001.crs)
print("Tipos geométricos:", trazado_r001.geom_type.unique().tolist())
print("Geometrías vacías:", trazado_r001.geometry.is_empty.sum())
print("Geometrías válidas:", trazado_r001.geometry.is_valid.all())

assert archivo_r001.exists()
assert len(trazado_r001) == 7
assert trazado_r001.crs.to_epsg() == 4326
assert set(trazado_r001.geom_type) == {"LineString"}
assert trazado_r001.geometry.is_valid.all()
assert not trazado_r001.geometry.is_empty.any()

print("Lectura del archivo original correcta")

display(
    trazado_r001[
        ["@id", "highway", "name", "surface", "@timestamp"]
    ]
)

Archivo: R001_rucu_pichincha_osm_2026-09-30.geojson
Número de geometrías: 7
Sistema de coordenadas: EPSG:4326
Tipos geométricos: ['LineString']
Geometrías vacías: 0
Geometrías válidas: True
Lectura del archivo original correcta


,@id,highway,name,surface,@timestamp
0,way/377339148,footway,Sendero Cumbre Rucu Pichincha,earth,2026-02-17 11:09:32+00:00
1,way/377342076,footway,Sendero Cumbre Rucu Pichincha,dirt,2025-08-26 16:56:41+00:00
2,way/377342086,path,Sendero Cumbre Rucu Pichincha,dirt,2025-08-26 16:56:41+00:00
3,way/377342089,footway,Sendero Cumbre Rucu Pichincha,dirt,2023-09-01 01:26:46+00:00
4,way/377354725,footway,Sendero Cumbre Rucu Pichincha,ground,2026-02-17 11:09:32+00:00
5,way/377359598,path,Sendero Cumbre Rucu Pichincha,earth,2026-02-17 11:09:32+00:00
6,way/442211574,path,Sendero Cumbre Rucu Pichincha,earth,2026-02-17 11:09:32+00:00


In [2]:
from shapely.ops import unary_union, linemerge

# Unir los segmentos únicamente en memoria
geometria_unida = linemerge(
    unary_union(trazado_r001.geometry.tolist())
)

trazado_unido_r001 = gpd.GeoDataFrame(
    {"id_ruta": ["R001"]},
    geometry=[geometria_unida],
    crs=trazado_r001.crs
)

# Proyectar a UTM 17 Sur para medir en metros
trazado_utm_r001 = trazado_unido_r001.to_crs(epsg=32717)

distancia_subida_km = (
    trazado_utm_r001.geometry.length.iloc[0] / 1000
)

distancia_ida_vuelta_km = distancia_subida_km * 2

print("Geometría resultante:", geometria_unida.geom_type)
print("Trazado continuo:", geometria_unida.geom_type == "LineString")
print("Distancia de subida:", round(distancia_subida_km, 3), "km")
print(
    "Distancia preliminar de ida y vuelta:",
    round(distancia_ida_vuelta_km, 3),
    "km"
)
print(
    "Diferencia frente a los 10 km publicados:",
    round(10.0 - distancia_ida_vuelta_km, 3),
    "km"
)

assert geometria_unida.geom_type == "LineString"
assert distancia_subida_km > 0

print("Los siete segmentos forman un trazado continuo")

Geometría resultante: LineString
Trazado continuo: True
Distancia de subida: 4.294 km
Distancia preliminar de ida y vuelta: 8.588 km
Diferencia frente a los 10 km publicados: 1.412 km
Los siete segmentos forman un trazado continuo


In [3]:
import pandas as pd
from shapely.geometry import Point

coordenadas = list(geometria_unida.coords)

extremo_a = Point(coordenadas[0])
extremo_b = Point(coordenadas[-1])

tabla_extremos = pd.DataFrame(
    {
        "extremo": ["A", "B"],
        "latitud": [extremo_a.y, extremo_b.y],
        "longitud": [extremo_a.x, extremo_b.x]
    }
)

display(tabla_extremos)

print(
    "El extremo A se encuentra junto a la cumbre del Rucu Pichincha."
)
print(
    "El extremo B corresponde al inicio del trazado OSM registrado."
)
print(
    "Se revisará si entre Cruz Loma y el extremo B falta un tramo inicial."
)

,extremo,latitud,longitud
0,A,-0.162188,-78.564296
1,B,-0.185758,-78.537522


El extremo A se encuentra junto a la cumbre del Rucu Pichincha.
El extremo B corresponde al inicio del trazado OSM registrado.
Se revisará si entre Cruz Loma y el extremo B falta un tramo inicial.


In [4]:
import pandas as pd
from shapely.ops import unary_union, linemerge

archivo_conexion = (
    proyecto
    / "02_datos_originales"
    / "trazados"
    / "R001_conexion_cruz_loma_osm_2026-09-30.geojson"
)

# Leer el archivo complementario sin modificarlo
conexion_r001 = gpd.read_file(archivo_conexion)

# Combinar en memoria los 7 segmentos iniciales y los 3 complementarios
segmentos_r001 = gpd.GeoDataFrame(
    pd.concat(
        [trazado_r001, conexion_r001],
        ignore_index=True
    ),
    crs=trazado_r001.crs
)

geometria_r001 = linemerge(
    unary_union(segmentos_r001.geometry)
)

# Calcular la longitud en UTM 17S
geometria_utm = gpd.GeoSeries(
    [geometria_r001],
    crs=segmentos_r001.crs
).to_crs(epsg=32717)

distancia_subida_km = geometria_utm.length.iloc[0] / 1000
distancia_ida_vuelta_km = distancia_subida_km * 2

print("Segmentos iniciales:", len(trazado_r001))
print("Segmentos complementarios:", len(conexion_r001))
print("Total de segmentos:", len(segmentos_r001))
print("Geometría resultante:", geometria_r001.geom_type)
print("Geometría válida:", geometria_r001.is_valid)
print("Distancia de subida:", round(distancia_subida_km, 3), "km")
print(
    "Distancia preliminar de ida y vuelta:",
    round(distancia_ida_vuelta_km, 3),
    "km"
)

assert archivo_conexion.exists()
assert len(conexion_r001) == 3
assert len(segmentos_r001) == 10
assert segmentos_r001["@id"].is_unique
assert geometria_r001.geom_type == "LineString"
assert geometria_r001.is_valid

print("Conexión con Cruz Loma validada correctamente")

Segmentos iniciales: 7
Segmentos complementarios: 3
Total de segmentos: 10
Geometría resultante: LineString
Geometría válida: True
Distancia de subida: 4.436 km
Distancia preliminar de ida y vuelta: 8.872 km
Conexión con Cruz Loma validada correctamente


### Resultado de la inspección de R001

Los siete segmentos iniciales y los tres segmentos complementarios forman un trazado continuo y válido desde el complejo del TelefériQo en Cruz Loma hasta un extremo próximo a la cumbre del Rucu Pichincha.

La longitud calculada inicialmente es de 4.436 km por trayecto y 8.872 km de ida y vuelta. La diferencia frente a los 10 km informados por la fuente turística motivó una revisión posterior. Durante la incorporación de elevación se comprobó que faltaba la vía OSM 377339146 entre este extremo y el nodo de la cumbre; la corrección se documenta al final del notebook.

Los archivos originales se conservan sin modificaciones. La geometría unificada se guardará posteriormente en `03_datos_intermedios`.

In [5]:
from shapely.geometry import LineString, Point

# Orientar el trazado desde Cruz Loma hacia la cumbre
coordenadas = list(geometria_r001.coords)
referencia_cruz_loma = Point(-78.5373223, -0.1866873)

if Point(coordenadas[-1]).distance(referencia_cruz_loma) < Point(
    coordenadas[0]
).distance(referencia_cruz_loma):
    coordenadas.reverse()

geometria_subida = LineString(coordenadas)

# Reconstruir la geometría en memoria y contrastar el producto archivado
carpeta_intermedia = (
    proyecto
    / "03_datos_intermedios"
    / "trazados"
)
assert carpeta_intermedia.is_dir()

archivo_intermedio = (
    carpeta_intermedia
    / "R001_trazado_subida_continuo_2026-10-01.geojson"
)

trazado_intermedio = gpd.GeoDataFrame(
    {
        "id_ruta": ["R001"],
        "tramo": ["subida"],
        "origen": ["Cruz Loma"],
        "destino": ["Extremo inicial próximo a la cumbre"],
        "distancia_km": [round(distancia_subida_km, 3)],
    },
    geometry=[geometria_subida],
    crs=segmentos_r001.crs,
)

assert archivo_intermedio.exists()

# Leer el archivo existente sin reescribirlo
comprobacion = gpd.read_file(archivo_intermedio)

print("Archivo comprobado:", archivo_intermedio.name)
print("Número de geometrías:", len(comprobacion))
print("Tipo geométrico:", comprobacion.geometry.iloc[0].geom_type)
print("Sistema de coordenadas:", comprobacion.crs)
print("Distancia registrada:", comprobacion.loc[0, "distancia_km"], "km")

assert archivo_intermedio.exists()
assert len(comprobacion) == 1
assert comprobacion.crs.to_epsg() == 4326
assert comprobacion.geometry.iloc[0].geom_type == "LineString"
assert comprobacion.geometry.iloc[0].is_valid
assert comprobacion.geometry.iloc[0].equals(geometria_subida)

print("Trazado intermedio de R001 comprobado correctamente")

Archivo comprobado: R001_trazado_subida_continuo_2026-10-01.geojson
Número de geometrías: 1
Tipo geométrico: LineString
Sistema de coordenadas: EPSG:4326
Distancia registrada: 4.436 km
Trazado intermedio de R001 comprobado correctamente


## R003 – Sendero Cascada Cóndor Machay

El archivo original se descargó directamente desde la API oficial de OpenStreetMap debido a la saturación temporal de los servidores Overpass. El extracto conserva sin modificaciones los elementos OSM del sector comprendido entre el parqueadero y la Cascada Cóndor Machay.

En esta sección se filtrarán los 25 segmentos peatonales que forman la conexión entre el parqueadero y la cascada.

In [6]:
import xml.etree.ElementTree as ET
from shapely.geometry import LineString
from shapely.ops import unary_union, linemerge

archivo_r003 = (
    proyecto
    / "02_datos_originales"
    / "trazados"
    / "R003_condor_machay_osm_2026-10-01.osm"
)

ids_r003 = {
    "1234489578", "1234489579", "1234489583",
    "1234489584", "1234489585", "1234489586",
    "1234489580", "1234489581", "1234489587",
    "1234489588", "1234489589", "1234489590",
    "1234489591", "1234489592", "1234489594",
    "1234489595", "1234489596", "1234489597",
    "1234489598", "1234489603", "1234489602",
    "1234489605", "1234489604", "1234489599",
    "1075545956",
}

raiz_r003 = ET.parse(archivo_r003).getroot()

coordenadas_nodos = {
    nodo.attrib["id"]: (
        float(nodo.attrib["lon"]),
        float(nodo.attrib["lat"]),
    )
    for nodo in raiz_r003.findall("node")
}

registros_r003 = []

for via in raiz_r003.findall("way"):
    id_via = via.attrib["id"]

    if id_via not in ids_r003:
        continue

    etiquetas = {
        etiqueta.attrib["k"]: etiqueta.attrib["v"]
        for etiqueta in via.findall("tag")
    }

    referencias = [
        nodo.attrib["ref"]
        for nodo in via.findall("nd")
    ]

    geometria = LineString(
        [coordenadas_nodos[ref] for ref in referencias]
    )

    registros_r003.append(
        {
            "@id": f"way/{id_via}",
            "highway": etiquetas.get("highway"),
            "name": etiquetas.get("name"),
            "geometry": geometria,
        }
    )

segmentos_r003 = gpd.GeoDataFrame(
    registros_r003,
    geometry="geometry",
    crs="EPSG:4326",
)

geometria_r003 = linemerge(
    unary_union(segmentos_r003.geometry)
)

geometria_r003_utm = gpd.GeoSeries(
    [geometria_r003],
    crs=segmentos_r003.crs,
).to_crs(epsg=32717)

distancia_r003_km = geometria_r003_utm.length.iloc[0] / 1000

print("Archivo:", archivo_r003.name)
print("Segmentos filtrados:", len(segmentos_r003))
print("Geometría resultante:", geometria_r003.geom_type)
print("Geometría válida:", geometria_r003.is_valid)
print("Distancia de ida:", round(distancia_r003_km, 3), "km")
print(
    "Distancia preliminar de ida y vuelta:",
    round(distancia_r003_km * 2, 3),
    "km"
)

assert archivo_r003.exists()
assert len(segmentos_r003) == 25
assert segmentos_r003["@id"].is_unique
assert set(segmentos_r003["highway"]) == {"path"}
assert geometria_r003.geom_type == "LineString"
assert geometria_r003.is_valid

print("Trazado de R003 validado correctamente")

Archivo: R003_condor_machay_osm_2026-10-01.osm
Segmentos filtrados: 25
Geometría resultante: LineString
Geometría válida: True
Distancia de ida: 3.742 km
Distancia preliminar de ida y vuelta: 7.484 km
Trazado de R003 validado correctamente


### Resultado de la inspección de R003

Los 25 segmentos peatonales filtrados forman un trazado continuo y válido entre el Parqueadero Cascada Cóndor Machay y la cascada.

La longitud calculada es de 3.742 km por trayecto y 7.484 km para el recorrido de ida y vuelta. El extracto original de OpenStreetMap se conserva sin modificaciones.

In [7]:
from shapely.geometry import Point

# Orientar el trazado desde el parqueadero hacia la cascada
coordenadas = list(geometria_r003.coords)
referencia_parqueadero = Point(-78.4228702, -0.4548333)

if Point(coordenadas[-1]).distance(referencia_parqueadero) < Point(
    coordenadas[0]
).distance(referencia_parqueadero):
    coordenadas.reverse()

geometria_ida_r003 = LineString(coordenadas)

carpeta_intermedia = (
    proyecto
    / "03_datos_intermedios"
    / "trazados"
)
assert carpeta_intermedia.is_dir()

archivo_intermedio_r003 = (
    carpeta_intermedia
    / "R003_trazado_ida_continuo_2026-10-01.geojson"
)

trazado_intermedio_r003 = gpd.GeoDataFrame(
    {
        "id_ruta": ["R003"],
        "tramo": ["ida"],
        "origen": ["Parqueadero Cascada Cóndor Machay"],
        "destino": ["Cascada Cóndor Machay"],
        "distancia_km": [round(distancia_r003_km, 3)],
    },
    geometry=[geometria_ida_r003],
    crs=segmentos_r003.crs,
)

assert archivo_intermedio_r003.exists()

comprobacion_r003 = gpd.read_file(
    archivo_intermedio_r003
)

print("Archivo comprobado:", archivo_intermedio_r003.name)
print("Número de geometrías:", len(comprobacion_r003))
print("Tipo geométrico:", comprobacion_r003.geometry.iloc[0].geom_type)
print("Sistema de coordenadas:", comprobacion_r003.crs)
print(
    "Distancia registrada:",
    comprobacion_r003.loc[0, "distancia_km"],
    "km"
)

assert archivo_intermedio_r003.exists()
assert len(comprobacion_r003) == 1
assert comprobacion_r003.crs.to_epsg() == 4326
assert comprobacion_r003.geometry.iloc[0].geom_type == "LineString"
assert comprobacion_r003.geometry.iloc[0].is_valid
assert comprobacion_r003.geometry.iloc[0].equals(geometria_ida_r003)

print("Trazado intermedio de R003 comprobado correctamente")

Archivo comprobado: R003_trazado_ida_continuo_2026-10-01.geojson
Número de geometrías: 1
Tipo geométrico: LineString
Sistema de coordenadas: EPSG:4326
Distancia registrada: 3.742 km
Trazado intermedio de R003 comprobado correctamente


## R005 – Sendero Tarabita - Cascada Nambillo

El extracto original se descargó directamente desde la API oficial de OpenStreetMap. La revisión identificó dos vías peatonales conectadas: Ruta 1 y Ruta 3.

El trazado se analizará como conexión peatonal preliminar. Se comprobará también su separación respecto al punto OSM de la Cascada Nambillo, ya que dicho punto está registrado sobre el río y no forma parte directa de una vía peatonal.

In [8]:
import xml.etree.ElementTree as ET
from shapely.geometry import LineString, Point
from shapely.ops import unary_union, linemerge

archivo_r005 = (
    proyecto
    / "02_datos_originales"
    / "trazados"
    / "R005_nambillo_osm_2026-10-01.osm"
)

ids_r005 = {"964152743", "196510810"}

raiz_r005 = ET.parse(archivo_r005).getroot()

coordenadas_nodos_r005 = {
    nodo.attrib["id"]: (
        float(nodo.attrib["lon"]),
        float(nodo.attrib["lat"]),
    )
    for nodo in raiz_r005.findall("node")
}

registros_r005 = []

for via in raiz_r005.findall("way"):
    id_via = via.attrib["id"]

    if id_via not in ids_r005:
        continue

    etiquetas = {
        etiqueta.attrib["k"]: etiqueta.attrib["v"]
        for etiqueta in via.findall("tag")
    }

    referencias = [
        nodo.attrib["ref"]
        for nodo in via.findall("nd")
    ]

    registros_r005.append(
        {
            "@id": f"way/{id_via}",
            "highway": etiquetas.get("highway"),
            "name": etiquetas.get("name"),
            "geometry": LineString(
                [coordenadas_nodos_r005[ref] for ref in referencias]
            ),
        }
    )

segmentos_r005 = gpd.GeoDataFrame(
    registros_r005,
    geometry="geometry",
    crs="EPSG:4326",
)

geometria_r005 = linemerge(
    unary_union(segmentos_r005.geometry)
)

geometria_r005_utm = gpd.GeoSeries(
    [geometria_r005],
    crs=segmentos_r005.crs,
).to_crs(epsg=32717)

punto_cascada = Point(
    coordenadas_nodos_r005["5309163322"]
)

punto_cascada_utm = gpd.GeoSeries(
    [punto_cascada],
    crs="EPSG:4326",
).to_crs(epsg=32717).iloc[0]

distancia_r005_km = (
    geometria_r005_utm.length.iloc[0] / 1000
)

separacion_cascada_m = (
    geometria_r005_utm.iloc[0].distance(
        punto_cascada_utm
    )
)

print("Archivo:", archivo_r005.name)
print("Segmentos filtrados:", len(segmentos_r005))
print("Nombres:", segmentos_r005["name"].tolist())
print("Geometría resultante:", geometria_r005.geom_type)
print("Geometría válida:", geometria_r005.is_valid)
print("Distancia de ida:", round(distancia_r005_km, 3), "km")
print(
    "Distancia preliminar de ida y vuelta:",
    round(distancia_r005_km * 2, 3),
    "km"
)
print(
    "Separación mínima hasta la cascada:",
    round(separacion_cascada_m, 1),
    "m"
)

assert archivo_r005.exists()
assert len(segmentos_r005) == 2
assert segmentos_r005["@id"].is_unique
assert set(segmentos_r005["highway"]) == {"footway"}
assert geometria_r005.geom_type == "LineString"
assert geometria_r005.is_valid
assert separacion_cascada_m < 150

print("Trazado preliminar de R005 validado")

Archivo: R005_nambillo_osm_2026-10-01.osm
Segmentos filtrados: 2
Nombres: ['Ruta 3', 'Ruta 1']
Geometría resultante: LineString
Geometría válida: True
Distancia de ida: 0.951 km
Distancia preliminar de ida y vuelta: 1.903 km
Separación mínima hasta la cascada: 106.5 m


Trazado preliminar de R005 validado


### Resultado de la inspección de R005

Las vías Ruta 1 y Ruta 3 forman una geometría LineString continua y válida. La longitud calculada es de 0.951 km por trayecto y 1.903 km para la ida y vuelta.

El trazado pasa a 106.5 m del punto OSM de la Cascada Nambillo. Por esta razón, su calidad preliminar se considera dudosa y deberá contrastarse con otra fuente antes de decidir su inclusión definitiva. No se añadirá una conexión artificial.

In [9]:
# Orientar desde la tarabita hacia el sector de la cascada
coordenadas = list(geometria_r005.coords)

referencia_tarabita = Point(
    coordenadas_nodos_r005["352972925"]
)

if Point(coordenadas[-1]).distance(referencia_tarabita) < Point(
    coordenadas[0]
).distance(referencia_tarabita):
    coordenadas.reverse()

geometria_preliminar_r005 = LineString(coordenadas)

carpeta_intermedia = (
    proyecto
    / "03_datos_intermedios"
    / "trazados"
)
assert carpeta_intermedia.is_dir()

archivo_intermedio_r005 = (
    carpeta_intermedia
    / "R005_trazado_preliminar_2026-10-01.geojson"
)

trazado_intermedio_r005 = gpd.GeoDataFrame(
    {
        "id_ruta": ["R005"],
        "tramo": ["ida"],
        "origen": ["Tarabita"],
        "destino": ["Sector Cascada Nambillo"],
        "estado": ["preliminar"],
        "distancia_km": [round(distancia_r005_km, 3)],
        "separacion_cascada_m": [
            round(separacion_cascada_m, 1)
        ],
    },
    geometry=[geometria_preliminar_r005],
    crs=segmentos_r005.crs,
)

assert archivo_intermedio_r005.exists()

comprobacion_r005 = gpd.read_file(
    archivo_intermedio_r005
)

print("Archivo comprobado:", archivo_intermedio_r005.name)
print("Número de geometrías:", len(comprobacion_r005))
print("Tipo geométrico:", comprobacion_r005.geometry.iloc[0].geom_type)
print("Estado:", comprobacion_r005.loc[0, "estado"])
print(
    "Distancia registrada:",
    comprobacion_r005.loc[0, "distancia_km"],
    "km"
)
print(
    "Separación registrada:",
    comprobacion_r005.loc[0, "separacion_cascada_m"],
    "m"
)

assert archivo_intermedio_r005.exists()
assert len(comprobacion_r005) == 1
assert comprobacion_r005.crs.to_epsg() == 4326
assert comprobacion_r005.geometry.iloc[0].geom_type == "LineString"
assert comprobacion_r005.geometry.iloc[0].is_valid
assert comprobacion_r005.geometry.iloc[0].equals(geometria_preliminar_r005)
assert comprobacion_r005.loc[0, "estado"] == "preliminar"

print("Trazado intermedio preliminar de R005 comprobado correctamente")

Archivo comprobado: R005_trazado_preliminar_2026-10-01.geojson
Número de geometrías: 1
Tipo geométrico: LineString
Estado: preliminar
Distancia registrada: 0.951 km
Separación registrada: 106.5 m
Trazado intermedio preliminar de R005 comprobado correctamente


## R006 – Sendero al Ilaló

El extracto original se descargó desde la API oficial de OpenStreetMap. La vía 198954451 contiene tanto el nodo 12867182981 (`Inicio del sendero`) como el nodo 12867182975 (`CIMA DEL SENDERO ILALO`). En esta sección se extrae únicamente el subtramo comprendido entre esos dos puntos, sin modificar el archivo original.

In [10]:
import xml.etree.ElementTree as ET
from shapely.geometry import LineString

archivo_r006 = (
    proyecto
    / "02_datos_originales"
    / "trazados"
    / "R006_ilalo_osm_2026-10-05.osm"
)

raiz_r006 = ET.parse(archivo_r006).getroot()
coordenadas_nodos_r006 = {
    nodo.attrib["id"]: (
        float(nodo.attrib["lon"]),
        float(nodo.attrib["lat"]),
    )
    for nodo in raiz_r006.findall("node")
}

id_via_r006 = "198954451"
nodo_inicio_r006 = "12867182981"
nodo_cima_r006 = "12867182975"

via_r006 = next(
    via
    for via in raiz_r006.findall("way")
    if via.attrib["id"] == id_via_r006
)
referencias_r006 = [nd.attrib["ref"] for nd in via_r006.findall("nd")]
indice_inicio_r006 = referencias_r006.index(nodo_inicio_r006)
indice_cima_r006 = referencias_r006.index(nodo_cima_r006)
referencias_tramo_r006 = referencias_r006[
    indice_inicio_r006:indice_cima_r006 + 1
]
geometria_r006 = LineString(
    [coordenadas_nodos_r006[ref] for ref in referencias_tramo_r006]
)

trazado_r006 = gpd.GeoDataFrame(
    {
        "id_ruta": ["R006"],
        "nombre": ["Sendero al Ilaló"],
        "id_via_osm": [id_via_r006],
        "nodo_inicio": [nodo_inicio_r006],
        "nodo_fin": [nodo_cima_r006],
        "estado": ["continuo"],
        "fecha": ["2026-10-05"],
    },
    geometry=[geometria_r006],
    crs="EPSG:4326",
)
distancia_r006_km = float(trazado_r006.to_crs(32717).length.iloc[0] / 1000)
trazado_r006["distancia_km"] = round(distancia_r006_km, 3)

archivo_intermedio_r006 = (
    proyecto
    / "03_datos_intermedios"
    / "trazados"
    / "R006_trazado_ida_continuo_2026-10-05.geojson"
)
assert archivo_intermedio_r006.exists()
comprobacion_r006 = gpd.read_file(archivo_intermedio_r006)

print("Archivo original:", archivo_r006.name)
print("Nodos del extracto:", len(raiz_r006.findall("node")))
print("Vías del extracto:", len(raiz_r006.findall("way")))
print("Vía utilizada:", id_via_r006)
print("Nodos del tramo:", len(referencias_tramo_r006))
print("Geometría resultante:", geometria_r006.geom_type)
print("Geometría válida:", geometria_r006.is_valid)
print("Distancia de ida:", round(distancia_r006_km, 3), "km")
print("Distancia preliminar de ida y vuelta:", round(distancia_r006_km * 2, 3), "km")

assert archivo_r006.exists()
assert len(referencias_tramo_r006) == 36
assert comprobacion_r006.crs.to_epsg() == 4326
assert comprobacion_r006.geometry.iloc[0].geom_type == "LineString"
assert comprobacion_r006.geometry.iloc[0].is_valid
assert comprobacion_r006.geometry.iloc[0].equals(geometria_r006)

print("Trazado intermedio de R006 comprobado y validado correctamente")

Archivo original:

 R006_ilalo_osm_2026-10-05.osm
Nodos del extracto: 2346
Vías del extracto: 65
Vía utilizada: 198954451
Nodos del tramo: 36
Geometría resultante: LineString
Geometría válida: True
Distancia de ida: 2.616 km
Distancia preliminar de ida y vuelta: 5.232 km
Trazado intermedio de R006 comprobado y validado correctamente


### Resultado de la inspección de R006

El subtramo de 36 nodos de la vía OSM 198954451 forma una geometría LineString continua y válida entre el inicio registrado del sendero y la cima. La longitud calculada es de 2.616 km por trayecto y 5.232 km para la ida y vuelta preliminar.

La calidad preliminar del trazado se considera apta. La elevación, el desnivel positivo acumulado y la pendiente se incorporarán en la siguiente fase.

## Revisión complementaria de R001 – conexión con la cumbre

Al incorporar Copernicus DEM se observó que el extremo del trazado inicial registraba cerca de 4473 m, mientras el nodo OSM de la cumbre está etiquetado con 4696 m. La revisión confirmó que faltaba la vía 377339146, que conecta directamente el extremo de la vía 442211574 con el nodo 249414034 de la cumbre.

In [11]:
archivo_r001_final = proyecto / "03_datos_intermedios" / "trazados" / "R001_trazado_subida_cumbre_2026-10-05.geojson"
assert archivo_r001_final.exists()
r001_final = gpd.read_file(archivo_r001_final)
assert len(r001_final) == 1
assert r001_final.geometry.iloc[0].geom_type == "LineString"
assert r001_final.geometry.iloc[0].is_valid
distancia_final_km = float(r001_final.to_crs(32717).length.iloc[0] / 1000)
assert abs(distancia_final_km - 4.804) < 0.01
print("R001 final verificada sin reescribir el trazado:", round(distancia_final_km, 3), "km de subida")

R001 final verificada sin reescribir el trazado: 4.804 km de subida


### Resultado corregido de R001

R001 queda formada por once vías OSM y alcanza el nodo documentado de la cumbre. La distancia corregida es de 4.804 km de subida y 9.607 km de ida y vuelta. El archivo inicial de 4.436 km se conserva como evidencia del proceso de revisión, pero los análisis posteriores utilizan `R001_trazado_subida_cumbre_2026-10-05.geojson`.